# Chess RL Engine — Training Analysis

Use this notebook to visualise training progress, analyse games, and debug the engine.

**Run this from the project root:**
```
cd chess-rl-engine
jupyter notebook notebooks/analysis.ipynb
```

In [ ]:
import sys
sys.path.insert(0, '..')  # Add project root to path

import json
import numpy as np
import matplotlib.pyplot as plt
import chess
import chess.svg
from IPython.display import SVG, display

from utils.visualiser import plot_training_curves, plot_elo_distribution
from rl_agent.agent import ChessAgent
from chess_engine.board import board_to_tensor, get_legal_move_mask
from chess_engine.game import play_game

print('Imports OK')

## 1. Training Curves

Load the metrics saved during training and plot ELO progression, win rates, and loss curves.

In [ ]:
# Plot training curves from saved metrics
# Change the path to match your logs directory
METRICS_PATH = '../logs/metrics_final.json'

try:
    plot_training_curves(METRICS_PATH)
except FileNotFoundError:
    print(f'Metrics file not found at {METRICS_PATH}')
    print('Train the engine first: python scripts/train.py')

## 2. Load a Trained Agent and Watch It Play

In [ ]:
import torch

CHECKPOINT_PATH = '../checkpoints/best_agent.pt'

device = torch.device('cpu')

try:
    agent = ChessAgent(device=device)
    agent.load(CHECKPOINT_PATH)
    print(f'Agent loaded. ELO: {agent.elo:.0f}')
except FileNotFoundError:
    print(f'Checkpoint not found at {CHECKPOINT_PATH}')
    print('Train the engine first, or update the path above.')

In [ ]:
# Play a game and display each position
# The agent plays itself (white and black)

board = chess.Board()
moves_made = []

for move_num in range(50):  # Show first 50 moves
    state = board_to_tensor(board)
    mask = get_legal_move_mask(board)
    
    # Use greedy action (best move) for cleaner display
    action = agent.select_best_action(state, mask)
    
    # Find the matching legal move
    from chess_engine.board import index_to_move
    move = None
    target_idx = action
    from chess_engine.board import move_to_index
    for legal_move in board.legal_moves:
        if move_to_index(legal_move) == target_idx:
            move = legal_move
            break
    
    if move is None:
        import random
        move = random.choice(list(board.legal_moves))
    
    board.push(move)
    moves_made.append(move.uci())
    
    if board.is_game_over():
        break

print(f'Game over after {len(moves_made)} moves')
print(f'Outcome: {board.outcome()}')
print(f'Moves: {" ".join(moves_made)}')

# Display final position
display(SVG(chess.svg.board(board, size=400)))

## 3. Raw Metrics Inspection

In [ ]:
try:
    with open(METRICS_PATH) as f:
        metrics = json.load(f)
    
    print(f"Total generations trained: {len(metrics['generation'])}")
    print(f"Best ELO achieved:         {max(metrics['best_elo']):.0f}")
    print(f"Final mean ELO:            {metrics['mean_elo'][-1]:.0f}")
    print(f"Final win rate:            {metrics['win_rate'][-1]:.1%}")
    print(f"Final entropy:             {metrics['entropy'][-1]:.4f}")
except Exception as e:
    print(f'Could not load metrics: {e}')